# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
import os
print(f'Path [{os.environ["PATH"]}]')

# torch_test.py
import torch
print(f"✅ Torch CUDA available: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# tf_test.py
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
import itertools 

# Pour les modèles et leur preprocessing
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# Pour la visualisation des performances
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pour instancier une couche Dense et modèle séquentiel
from tensorflow.keras import Sequential, callbacks
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Rescaling
from tensorflow.keras.datasets import mnist

# 2. Loading and Data Enrichment

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
# Affichage aléatoire de 6 images
fig, axs = plt.subplots(2, 3, figsize=(14,9))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(np.arange(0, len(y_train)), size=6)):
    img = X_train[i]
    axs[j].axis('off')
    # Affichage de l'image en niveaux de gris
    axs[j].imshow(img, cmap='gray', interpolation='none')
    # Titre avec le label
    axs[j].set_title(f'Label: {str(y_train[i])}')
plt.show()

In [ ]:
# image moyenne d'un label (moyenne par pixel de sur toutes les lignes d'un label donné)
fig, axs = plt.subplots(2, 5, figsize=(14,6))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for i in range(10):
    # Sélection des lignes de X_train correspondant au label i
    t = X_train[y_train == i]
    # Calcul de l'image moyenne
    img = t.mean(axis=0)     
    # Affichage de l'image dans le i+1-ème emplacement d'une grille de figures
    # à 2 lignes et 5 colonnes.    
    axs[i].imshow(img, cmap='gray_r', interpolation='None')

In [ ]:
# écart type d'un label (moyenne par pixel de sur toutes les lignes d'un label donné)
fig, axs = plt.subplots(2, 5, figsize=(14,6))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for i in range(10):
    # Sélection des lignes de X_train correspondant au label i
    t = X_train[y_train == i]
    # Calcul de l'image moyenne
    img = t.std(axis=0)     
    # Affichage de l'image dans le i+1-ème emplacement d'une grille de figures
    # à 2 lignes et 5 colonnes.    
    axs[i].imshow(img, cmap='gray_r', interpolation='None')

In [ ]:
# Changer la forme de X_train et X_test
X_train_resh = X_train.reshape([-1, 28*28])
X_test_resh = X_test.reshape([-1, 28*28])

# Shape de X_train and y_train
print('Shape of X:', X_train_resh.shape)
print('Shape of y:', y_train.shape)

# 3. Deep learning

## 3.0 : (optionel) modèle de référence pour comparaison : RFC

In [ ]:
# instanciation d'un modèle de reference pour comparaison
clf_rfc = RandomForestClassifier(n_jobs=-1)
clf_rfc.fit(X_train_resh, y_train)

In [ ]:
# evaluation de ses performances
y_test_pred = clf_rfc.predict(X_test_resh)
print("Score accuracy:", clf_rfc.score(X_test_resh, y_test))
print("Evaluation détaillée de la Classification par RDF :\n \n" ,
      (classification_report(y_test, y_test_pred)))
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test, y_test_pred, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(8, 6))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
plt.ylabel('Vrais labels')
plt.xlabel('Labels prédits')
plt.show()

In [ ]:
# Analyse fine multi classe sur la matrice de confusion avec itertool
for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
    if cnf_matrix[i, j] > 0.015 and i != j:  # Seuil de 1.5% pour les erreurs significatives
        print(f"{cnf_matrix[i, j] * 100:.2f}% des instances du chiffre {i} ont été classées comme {j}.")

In [ ]:
# projection graphique des prédictions sur des images aléatoirement prises dans l'échantillon de test
fig, axs = plt.subplots(2, 3, figsize=(14,9))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(np.arange(0, len(y_test)), size=6)):
    img = X_test_resh[i]
    # redimensionnement pour l'affichage
    img = img.reshape(28, 28)
    axs[j].axis('off')
    # Affichage de l'image en niveaux de gris
    axs[j].imshow(img, cmap='gray', interpolation='none')
    # Titre avec le label et la prédiction
    axs[j].set_title(f'Label: réel[{str(y_test[i])}] | prédit[{str(y_test_pred[i])}]')
plt.show()

## 3.1 Modèle Tensor Flow Keras

#### Creation & Compilation

In [ ]:
# Création du modèle séquentiel
nn_tfkeras_seq = Sequential()

# Définitions des dimension d'entrée et de sortie optimisées
num_pixels = X_train_resh.shape[1]
num_classes = np.unique(y_test).size
print(f"num_pixels[{num_pixels}] num_classes[{num_classes}]")
# Ajout de la couche de rescaling comme première couche utilisation adaptée aux pixels (255 valeurs)
nn_tfkeras_seq.add(Rescaling(1./255, input_shape=(num_pixels,))) 
nn_tfkeras_seq.add(Dense(units=128, activation='relu'))
nn_tfkeras_seq.add(BatchNormalization())
nn_tfkeras_seq.add(Dense(units=512, activation='relu'))
nn_tfkeras_seq.add(Dropout(rate=0.2))
nn_tfkeras_seq.add(Dense(units=256, activation='relu'))
nn_tfkeras_seq.add(Dense(units=128, activation='relu'))
nn_tfkeras_seq.add(Dense(units=num_classes, activation='softmax'))

In [ ]:
nn_tfkeras_seq.summary()

In [ ]:
# callback optimisant le temps de traitement
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    mode='min',
    restore_best_weights=True
)
lr_plateau = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    patience=5,
    factor=0.8,
    verbose=2,
    mode='min'
)

In [ ]:
nn_tfkeras_seq.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

#### Entrainement et Prédiction

In [ ]:
train_history = nn_tfkeras_seq.fit(
    X_train_resh, 
    y_train, 
    epochs=50, 
    batch_size=32, 
    validation_split=0.2, 
    callbacks=[early_stopping,lr_plateau]
)

In [ ]:
# Prédictions du modèle : pour chaque échantillon, un vecteur de probabilités (1 par classe, grâce à softmax)
# Les classes sont ici codées de 0 à 9 (multi-classe)
y_test_prob = nn_tfkeras_seq.predict(X_test_resh)
# evaluation du modèle (perte et accuracy)
print(nn_tfkeras_seq.evaluate(X_test_resh, y_test))

In [ ]:
# les predictions peuvent servir en utilisant l'argmax par colonne (indice de la colonne ayant la plus forte proba)
# à déterminer la classe prédite (si colonne 0 = classe 0, colonne 1 = classe 1, etc...)
# vu qu'on a encodé y avec des label de 0 à X
y_test_pred = np.argmax(y_test_prob, axis=1)

#### Visualisation des résultats

In [ ]:
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test, y_test_pred, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(8, 6))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
plt.ylabel('Vrais labels')
plt.xlabel('Labels prédits')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,6))
axs[0].plot(train_history.history['loss'], label='Loss (entraînement)')
axs[0].plot(train_history.history['val_loss'], label='Loss (validation)')
axs[0].set_title('Courbe de la perte par époque')
axs[0].set_xlabel('Épochs')
axs[0].set_ylabel('Perte')
axs[0].legend()
axs[1].plot(train_history.history['accuracy'], label='Accuracy (entraînement)')
axs[1].plot(train_history.history['val_accuracy'], label='Accuracy (validation)')
axs[1].set_title('Courbe de l\'accuracy par époque')
axs[1].set_xlabel('Épochs')
axs[1].set_ylabel('Accuracy')
axs[1].legend()
plt.show()